# Demo 05 : Agent failures

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · tool-errors

We handle a tool that fails during execution.

In [2]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = data("game-studio/studio.db")
SECTIONS = []
for path in sorted(Path(data_folder("game-studio/docs")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]


def rows(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(row) for row in conn.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]

@tool
def game_info(title: str) -> str:
    """Look up a Comet Arcade game by its title or id. Returns id, title, genre, status and pitch."""
    t = title.strip()
    found = rows("SELECT game_id, title, genre, status, pitch FROM games WHERE game_id = ? OR title LIKE ?", t.upper(), f"%{t}%")
    return json.dumps(found[0]) if found else "Unknown game"

@tool
def assets_for_game(game_id: str) -> str:
    """List sprites, palettes and sounds for a game id such as CS-01, with colour and status."""
    return json.dumps(rows("SELECT asset_id, kind, name, colour, status FROM assets WHERE game_id = ?", game_id.strip().upper()))

@tool
def search_design_docs(query: str) -> str:
    """Search the design docs and the studio style guide. Returns the three best matching sections."""
    return "\n\n---\n\n".join(best(SECTIONS, query, 3))

tools = [game_info, assets_for_game, search_design_docs]
question = {"messages": [("human", "Playtesters lose Comet Snake to comets they cannot see. Look up the game first, then its assets, "
                                   "then the style guide section on the rule that applies. Which asset is at fault, and what should the Artist change?")]}


def show(result):
    for message in result["messages"]:
        print(message.type, str(getattr(message, "tool_calls", None) or message.content)[:150].replace("\n", " "))
    print("model calls:", sum(m.type == "ai" for m in result["messages"]))
    print("input tokens per call:", [m.usage_metadata["input_tokens"] for m in result["messages"] if m.type == "ai"])

In [3]:
from langchain.messages import AIMessage, HumanMessage, ToolCall
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel


class FakeToolModel(GenericFakeChatModel):
    def bind_tools(self, tools, **kwargs):
        return self

We inject a timeout of the playtest server. Without a handler the run raises. `ToolErrorMiddleware` returns selected execution errors as error messages, which the real model can use to report the unavailable service.

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolErrorMiddleware


@tool
def live_playtests(game_id: str) -> str:
    """Read the live playtest scores of a game id such as CS-01."""
    raise TimeoutError("playtest server unavailable")


fake = FakeToolModel(messages=iter([AIMessage("", tool_calls=[
    ToolCall(name="live_playtests", args={"game_id": "CS-01"}, id="scores-1")])]))
try:
    create_agent(fake, tools=[live_playtests]).invoke({"messages": [HumanMessage("Read the scores.")]})
except TimeoutError as exc:
    print("unhandled:", type(exc).__name__)


def on_error(exc, request):
    if isinstance(exc, TimeoutError):
        return "Playtest server unavailable. Do not guess a score. Ask for a manual playtest."
    return None


safe = create_agent(llm, tools=[live_playtests], middleware=[ToolErrorMiddleware(on_error)],
                    system_prompt="Use live_playtests for scores. If unavailable, report that and stop.")
out = safe.invoke({"messages": [("human", "Read the live scores of CS-01.")]}, {"recursion_limit": 16})
for message in out["messages"]:
    print(message.type, getattr(message, "status", ""), message.content or getattr(message, "tool_calls", []))

unhandled: TimeoutError


human  Read the live scores of CS-01.
ai  [{'name': 'live_playtests', 'args': {'game_id': 'CS-01'}, 'id': 'call_th0FyNsD1SvR7ewM5Dl2WDNk', 'type': 'tool_call'}]
tool error Playtest server unavailable. Do not guess a score. Ask for a manual playtest.
ai  The live playtest server is unavailable, so I can’t read the scores. Please provide a manual playtest.


## b · model-fallback

We fail the primary model locally, then use the course model as the fallback. The failure path runs without a request for an invalid model name.

In [5]:
from langchain.agents.middleware import ModelFallbackMiddleware


class UnavailableModel(FakeToolModel):
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        print("primary: unavailable")
        raise ConnectionError("injected model outage")


primary = UnavailableModel(messages=iter([]))
agent = create_agent(primary, tools=tools, middleware=[ModelFallbackMiddleware(llm)],
                     system_prompt="You are the studio assistant of Comet Arcade. Use the tools. Answer in three sentences.")
out = agent.invoke(question, {"recursion_limit": 16})
print("fallback answer:", out["messages"][-1].content)

primary: unavailable


primary: unavailable


primary: unavailable


primary: unavailable


fallback answer: The **comet sprite (CS-SPR-03)** is at fault: its draft blue (#3a4a7a) lacks sufficient contrast against the deep-space background (#0b1030). The style guide requires game-ending hazards to have at least 4.5:1 contrast and to use the warm danger colour; the Artist should keep the comet’s shape and change its colour to **#ff6b5e**.


## c · call-limits

We script repeated lookups to show both limits. Run limits reset per invocation. Thread limits add up when a checkpointer saves state.

In [6]:
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware


def repeated_lookup():
    return FakeToolModel(messages=iter([
        AIMessage("", tool_calls=[ToolCall(name="game_info", args={"title": "Comet Snake"}, id="1")]),
        AIMessage("", tool_calls=[ToolCall(name="game_info", args={"title": "Comet Snake"}, id="2")]),
        AIMessage("Finished.")]))


for middleware in [ModelCallLimitMiddleware(run_limit=1, exit_behavior="end"),
                   ToolCallLimitMiddleware(tool_name="game_info", run_limit=1, exit_behavior="end")]:
    out = create_agent(repeated_lookup(), tools=[game_info], middleware=[middleware]).invoke(question)
    print("limit:", type(middleware).__name__)
    for message in out["messages"]:
        print(message.type, str(message.content or getattr(message, "tool_calls", []))[:150])

limit: ModelCallLimitMiddleware
human Playtesters lose Comet Snake to comets they cannot see. Look up the game first, then its assets, then the style guide section on the rule that applies
ai [{'name': 'game_info', 'args': {'title': 'Comet Snake'}, 'id': '1', 'type': 'tool_call'}]
tool {"game_id": "CS-01", "title": "Comet Snake", "genre": "arcade", "status": "in development", "pitch": "Snake in space : eat stars, dodge comets, every 
ai Model call limits exceeded: run limit (1/1)
limit: ToolCallLimitMiddleware
human Playtesters lose Comet Snake to comets they cannot see. Look up the game first, then its assets, then the style guide section on the rule that applies
ai [{'name': 'game_info', 'args': {'title': 'Comet Snake'}, 'id': '1', 'type': 'tool_call'}]
tool {"game_id": "CS-01", "title": "Comet Snake", "genre": "arcade", "status": "in development", "pitch": "Snake in space : eat stars, dodge comets, every 
ai [{'name': 'game_info', 'args': {'title': 'Comet Snake'}, 'id': '2', 'type':

## d · runaway-agent

A tool that never reports success keeps a real model calling it. With a call limit the run ends with a message. Without one, `create_agent` allows thousands of graph steps, so we set a low `recursion_limit` here and catch the exception. Every call pays for the whole context again. Whether the model keeps polling can vary from run to run.

In [7]:
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.errors import GraphRecursionError


class Usage(BaseCallbackHandler):
    def __init__(self):
        self.inputs = []

    def on_llm_end(self, response, **kwargs):
        self.inputs.append(response.generations[0][0].message.usage_metadata["input_tokens"])


@tool
def build_status(game_id: str) -> str:
    """Read the status of the current build of a game id such as CS-01."""
    return "still building"


WATCHER = "You watch builds at Comet Arcade. Call build_status again until the build has finished, then report the result. Never stop before that."
task = {"messages": [("human", "Tell me when the build of CS-01 has finished.")]}

capped = create_agent(llm, tools=[build_status], system_prompt=WATCHER,
                      middleware=[ModelCallLimitMiddleware(run_limit=6, exit_behavior="end")])
usage = Usage()
out = capped.invoke(task, {"callbacks": [usage]})
print("with a limit of 6")
print("model calls:", len(usage.inputs))
print("input tokens per call:", usage.inputs)
print("total input tokens:", sum(usage.inputs))
print("last message:", out["messages"][-1].content)

uncapped = create_agent(llm, tools=[build_status], system_prompt=WATCHER)
print("default step budget of create_agent:", uncapped.config["recursion_limit"])
usage = Usage()
try:
    uncapped.invoke(task, {"callbacks": [usage], "recursion_limit": 25})
except GraphRecursionError as exc:
    print("stopped by:", type(exc).__name__)
print("model calls:", len(usage.inputs))
print("input tokens per call:", usage.inputs)
print("total input tokens:", sum(usage.inputs))

with a limit of 6
model calls: 6
input tokens per call: [98, 132, 166, 200, 234, 268]
total input tokens: 1098
last message: Model call limits exceeded: run limit (6/6)
default step budget of create_agent: 9999


stopped by: GraphRecursionError
model calls: 13
input tokens per call: [98, 132, 166, 200, 234, 268, 302, 336, 370, 404, 438, 472, 506]
total input tokens: 3926


## Things to try

Optional : we can change the game, the asset or the question and run the example again.